# Véletlenszerű játszma Stockfish-elemzése + Gemini narráció

**Pipeline:**
1. Véletlenszerű játszma sorsolása a `mychessdotcomgames.parquet` fájlból
2. Stockfish elemzés (centipawn annotációk lépésenként)
3. Annotált PGN exportálása → `output/pgns/elemzett.pgn`
4. Gemini API: didaktikus narráció generálása
5. Narráció mentése → `output/llm-answers/GeminiMondta.txt`

In [1]:
import os
import sys
import random
import shutil

# ROOT_DIR: a könyvtár ahol a config.py van
_cwd = os.getcwd()
ROOT_DIR = _cwd if os.path.exists(os.path.join(_cwd, "config.py"))            else os.path.abspath(os.path.join(_cwd, ".."))
sys.path.insert(0, ROOT_DIR)

import polars as pl
import chess
import chess.pgn
import chess.engine

import config
from src.llm_client import generate_text

print(f"ROOT_DIR    : {ROOT_DIR}")
print(f"LLM provider: {config.LLM_PROVIDER}  ({config.LLM_MODEL})")
print(f"API kulcs   : {chr(39)+chr(79)+chr(75)+chr(39) if config.LLM_API_KEY else chr(39)+'HIÁNYZIK – töltsd ki a secrets.py-t!'+chr(39)}")

ROOT_DIR    : D:\Workspace\chess-pgn-analysis
LLM provider: openai  (gpt-4o-mini)
API kulcs   : 'OK'


## 1. Véletlen játszma kiválasztása

In [2]:
PARQUET_PATH = os.path.join(ROOT_DIR, "output", "parquet", "mychessdotcomgames.parquet")

df = pl.read_parquet(PARQUET_PATH)
print(f"Összes játszma a parquet-ban: {len(df):,}")

idx = random.randint(0, len(df) - 1)
row = df.row(idx, named=True)

print(f"\nKiválasztott játszma (sor-index={idx}, game_id={row['game_id']}):")
print(f"  Fehér  : {row['white']} ({row['white_elo']})")
print(f"  Fekete : {row['black']} ({row['black_elo']})")
print(f"  Megnyitó: {row['eco']} · {row['opening']}")
print(f"  Eredmény: {row['result']}  |  Lépések: {row['num_moves']}")

Összes játszma a parquet-ban: 1,377

Kiválasztott játszma (sor-index=1277, game_id=1278):
  Fehér  : massenaldo (1073)
  Fekete : Wujajin (1074)
  Megnyitó: D00 · 
  Eredmény: 1-0  |  Lépések: 21


## 2. Stockfish elemzés

In [3]:
def find_stockfish() -> str:
    if getattr(config, 'STOCKFISH_PATH', None) and os.path.isfile(config.STOCKFISH_PATH):
        return config.STOCKFISH_PATH
    bundled = os.path.join(ROOT_DIR, "stockfish", "stockfish-windows-x86-64-avx2.exe")
    if os.path.isfile(bundled):
        return bundled
    found = shutil.which("stockfish")
    if found:
        return found
    raise FileNotFoundError("Stockfish nem található! Ellenőrizd a stockfish/ mappát.")

SF_PATH = find_stockfish()
print(f"Stockfish: {SF_PATH}")

Stockfish: D:\Workspace\chess-pgn-analysis\stockfish\stockfish-windows-x86-64-avx2.exe


In [4]:
import asyncio
from tqdm.notebook import tqdm

# Windows + Python 3.9 alatt a Jupyter SelectorEventLoop-ot használ,
# ami nem tud subprocesst indítani – a chess.engine ezt igényli.
if hasattr(asyncio, 'WindowsProactorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Depth csökkentése gyorsabb futáshoz (config: 18, teszt: 12)
DEPTH       = 12
MOVES_LIMIT = config.STOCKFISH_MOVES_LIMIT

moves_uci_list = row['moves_uci'].strip().split()
to_analyze     = min(len(moves_uci_list), MOVES_LIMIT)
print(f"Elemzés: mélység={DEPTH}, lépések={to_analyze}/{len(moves_uci_list)}\n")

board       = chess.Board()
evaluations = []

with chess.engine.SimpleEngine.popen_uci(SF_PATH) as engine:
    limit = chess.engine.Limit(depth=DEPTH)

    for uci in tqdm(moves_uci_list[:MOVES_LIMIT], desc="Stockfish", unit="lépés"):
        move = chess.Move.from_uci(uci)
        if move not in board.legal_moves:
            print(f"  Illegális lépés: {uci} – leállítás")
            break

        san = board.san(move)
        board.push(move)

        info  = engine.analyse(board, limit)
        score = info["score"].white()

        evaluations.append({
            "uci":  uci,
            "san":  san,
            "cp":   score.score(mate_score=10000),
            "mate": score.mate(),
        })

print(f"\nKész: {len(evaluations)} lépés elemezve.")

Elemzés: mélység=12, lépések=21/21



Stockfish:   0%|          | 0/21 [00:00<?, ?lépés/s]


Kész: 21 lépés elemezve.


## 3. Annotált PGN exportálása

In [5]:
PGN_DIR  = os.path.join(ROOT_DIR, "output", "pgns")
PGN_PATH = os.path.join(PGN_DIR, "elemzett.pgn")
os.makedirs(PGN_DIR, exist_ok=True)

game_pgn = chess.pgn.Game()
game_pgn.headers.update({
    "Event":    row.get("event",    "?"),
    "Site":     row.get("site",     "?"),
    "Date":     row.get("date",     "?"),
    "White":    row.get("white",    "?"),
    "Black":    row.get("black",    "?"),
    "Result":   row.get("result",   "*"),
    "WhiteElo": str(row.get("white_elo", "?")),
    "BlackElo": str(row.get("black_elo", "?")),
    "ECO":      row.get("eco",      "?"),
    "Opening":  row.get("opening",  "?"),
})

node = game_pgn
for ev in evaluations:
    node = node.add_variation(chess.Move.from_uci(ev["uci"]))
    if ev["mate"] is not None:
        node.comment = f"[%eval #{ev['mate']}]"
    elif ev["cp"] is not None:
        node.comment = f"[%eval {ev['cp'] / 100:.2f}]"

with open(PGN_PATH, "w", encoding="utf-8") as f:
    print(game_pgn, file=f, end="\n")

print(f"PGN elmentve: {PGN_PATH}")
with open(PGN_PATH, encoding="utf-8") as f:
    print("\nElőnézet (első 600 karakter):")
    print(f.read()[:600])

PGN elmentve: D:\Workspace\chess-pgn-analysis\output\pgns\elemzett.pgn

Előnézet (első 600 karakter):
[Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.03.21"]
[Round "?"]
[White "massenaldo"]
[Black "Wujajin"]
[Result "1-0"]
[WhiteElo "1073"]
[BlackElo "1074"]
[ECO "D00"]
[Opening ""]

1. d4 { [%eval 0.30] } 1... d5 { [%eval 0.38] } 2. e4 { [%eval -0.65] } 2... dxe4 { [%eval -0.53] } 3. Nc3 { [%eval -0.66] } 3... Nf6 { [%eval -0.64] } 4. f3 { [%eval -0.68] } 4... exf3 { [%eval -0.68] } 5. Qxf3 { [%eval -1.37] } 5... Bg4 { [%eval 0.76] } 6. Qxb7 { [%eval 0.52] } 6... Nbd7 { [%eval 0.49] } 7. Nb5 { [%eval 0.39] } 7... Rb8 { [%eval 4.58] } 8. Qc6 { [%eval 1.11] } 8... e6 { [%eval 2.73] } 9. Nx


## 4. Gemini narráció generálása

In [6]:
NARRATION_PROMPT = (
    "Készíts 2 bekezdésben didaktív szöveges narrációt a csatolt .pgn sakkjátszma alapján, "
    "ügyelj rá, hogy a figurák nevét jól írd ki (mivel hangosan fel lesz olvasva), "
    "amikor egy lépésre hivatkozol, pl. Rxh7-et így írd: bástya üti h7-et, stb.! "
    "A narrációd célja az oktatás és hogy döntő kulcspillanatokat kiemelj!"
)

with open(PGN_PATH, encoding="utf-8") as f:
    pgn_text = f.read()

full_prompt = NARRATION_PROMPT + chr(10) * 2 + pgn_text
narration = generate_text(full_prompt)

print("--- LLM narráció ---")
print(narration)

--- LLM narráció ---
A játszma kezdetén a fehér játékos egy klasszikus nyitással indít: a d4 lépésével megnyitja a centrumot. Ezt követően a fekete játékos a d5-öt játssza, amivel válaszol a fehér nyitásra. A fehér e4-et lép, megkísérelve még inkább megerősíteni a centrumot, de a fekete játékos gyorsan szembeszáll a kezdeményezéssel, hiszen dxe4 lépéssel elveszi a fehér gyalogot. Az újabb lépések során a fehér lovat (Nc3) és a fekete lovat (Nf6) mozgatják, míg a f3 lépéssel a fehér megpróbálja megerősíteni saját pozícióját. Ekkor a fekete figura – a gyalog – f3-at üti, amely a fehér királynak kissé helyzetbe hozza az irányítást.

A következő lépések kulcsfontosságú fordulatot hoznak. A fehér király (Qxf3) előre lép, hogy visszaszerezze az uralmat a mezők felett. A fekete játékos ezután a bástyát (Bg4) mozdítja, hogy nyomás alá helyezze a fehér királyt. A fehér viszont gyorsan válaszol, hiszen a bástyát (Qxb7) veszi el, amely hatalmas fölényt biztosít. A következő lépésben a fehér ló (N

## 5. Narráció mentése

In [7]:
LLM_DIR  = os.path.join(ROOT_DIR, "output", "llm-answers")
LLM_PATH = os.path.join(LLM_DIR, "GeminiMondta.txt")
os.makedirs(LLM_DIR, exist_ok=True)

with open(LLM_PATH, "w", encoding="utf-8") as f:
    f.write(narration)

print(f"Gemini válasz elmentve: {LLM_PATH}")

Gemini válasz elmentve: D:\Workspace\chess-pgn-analysis\output\llm-answers\GeminiMondta.txt
